# Exploratory Data Analysis

## Dataset Analysis

For this recommendation system will be used the MovieLens dataset, witch include information about user rating and tags for a wide movie range

## Subdataset

The data is divided into four dataset that contains:

**movies.csv** the full plataform movie catalog

**ratings.csv** Dataset that relates movie, user (both by id) and it's rating

**tags.csv** Dataset that relates user and it's given tags to a movie

### Importing Dataset

In [ ]:
! pip install pandas

In [1]:
! pip install -r ../requirements.txt

^C


In [2]:
# Importing libraries

import os
import sys
import pandas as pd

In [3]:
# Importing dataset

dataset_folder = os.path.join("..", "data", "raw")

file_path_movies = os.path.join(dataset_folder, 'movies.csv')
file_path_ratings = os.path.join(dataset_folder, 'ratings.csv')

In [5]:
df_movies = pd.read_csv(file_path_movies, encoding='utf-8')
df_ratings = pd.read_csv(file_path_ratings, encoding='utf-8')

### Dataset Info

In [6]:
# Main info about dataset

print(' ----- MOVIES -----')
print('INFO:')
df_movies.info()

print(f'SHAPE: {df_movies.shape}')

 ----- MOVIES -----
INFO:
<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int64
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 2.0 MB
SHAPE: (87585, 3)


In [7]:
print(' ----- RATING -----')
print('INFO:')
df_ratings.info()

print(f'SHAPE: {df_ratings.shape}')

 ----- RATING -----
INFO:
<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB
SHAPE: (32000204, 4)


In [14]:
# Validate duplicates

movie_duplicates = df_movies.duplicated().sum()
rating_duplicates = df_ratings.duplicated().sum()

print(f'Movie: {movie_duplicates}')
print(f'Rating: {rating_duplicates}')

Movie: 0
Rating: 0


### Merging Dataset

The rating dataset will be merged with the movie dataset to containg the relation between rating, movie and its gender, in order to have a complete information to create a recommendation system

In [12]:
# Merge both dataset into one

df_movie_rating = pd.merge(left=df_ratings, right=df_movies, how='left', on='movieId')

In [38]:
print(' ----- MOVIE RATING -----')
print('INFO:')
df_movie_rating.info()

print(f'SHAPE: {df_movie_rating.shape}')

 ----- MOVIE RATING -----
INFO:
<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 6 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
 4   title      str    
 5   genres     str    
dtypes: float64(1), int64(3), str(2)
memory usage: 1.4 GB
SHAPE: (32000204, 6)


In [13]:
df_movie_rating.head()

,userId,movieId,rating,timestamp,title,genres
0,1,17,4.0,944249077,Sense and Sensibility (1995),Drama|Romance
1,1,25,1.0,944250228,Leaving Las Vegas (1995),Drama|Romance
2,1,29,2.0,943230976,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
3,1,30,5.0,944249077,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Crime|Drama
4,1,32,5.0,943228858,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller


### Formating Merged Dataset

In [14]:
# Delete timestamp column
df_movie_rating.drop(columns=['timestamp'], inplace=True)

In [15]:
# Inspect rating column

df_movie_rating['rating'].value_counts()

rating
4.0    8367654
3.0    6054990
5.0    4596577
3.5    4290105
4.5    2974000
2.0    2028622
2.5    1685386
1.0     946675
1.5     531063
0.5     525132
Name: count, dtype: int64

In [ ]:
# Looking for empty columns

cols = ['genres', 'title']

for col in cols:
    empty_values = df_movie_rating[df_movie_rating[col].isna() | df_movie_rating[col].str.strip().eq('')]
    print(f'{col}: {empty_values}')


In [ ]:
# Dividing genres into more columns
genres = df_movie_rating['genres'].str.get_dummies(sep="|")
df_movie_rating_processed = pd.concat([df_movie_rating, genres], axis=1)
 

In [19]:
df_movie_rating_processed.head()

,userId,movieId,rating,title,genres,no_genre,Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,17,4.0,Sense and Sensibility (1995),Drama|Romance,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,1,25,1.0,Leaving Las Vegas (1995),Drama|Romance,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,1,29,2.0,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,0,0,1,0,0,...,0,0,0,0,1,0,1,0,0,0
3,1,30,5.0,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Crime|Drama,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,32,5.0,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,0,0,0,0,0,...,0,0,0,0,1,0,1,1,0,0


In [ ]:
# Rename column to ease later steps
df_movie_rating_processed.rename(columns={'(no genres listed)': 'no_genre'}, inplace=True)

In [21]:
# Drop Gender column
df_movie_rating_processed.drop(columns=['genres'], inplace=True)

In [22]:
# Create processed file

processed_file = os.path.join(dataset_folder, '..', 'processed', 'df_movie_rating.csv')
df_movie_rating_processed.to_csv(processed_file)